In [1]:
reviews = "../results/training_review_diverse"

In [2]:
import pandas as pd
df = pd.read_csv("../results/bench_scores_deepreview_flash.csv")

In [3]:
def get_gt(review_filename):
    paper_id = review_filename.split(".")[0]
    row = df[df["paper_id"] == paper_id]
    return row["gt_avg_score"]

In [4]:
rl_prompt = """
You will get a review of a paper and a set of retrieved anchor reviews, based on the anchor review, estimate the score of the paper under review. The score should be between 1 and 10, where 1 is the worst and 10 is the best. Round to the nearest .5 or .0. 

Scoring rules:
- Your final score must be positioned relative to the retrieved anchors.
- Do not pick a score first and then justify it. Compare to anchors first, let the comparison set the score.
- The number of weaknesses listed is not a signal for a bad paper — focus on weakness content and anchor scores.
- Score distribution: extreme scores are rare but valid. If the paper truly is exceptional or truly weak, give an extreme score even if most retrieved anchors sit in the middle.
- Do NOT cluster scores around 5, the score should be relative to the retrieval samples. Score a good paper high and a bad paper low. 
- Compare the paper under review with every single anchor paper

Scoring scale:
1 - strong reject
3 - reject
4 - borderline reject
6 - borderline accept
8 - accept
10 - strong accept

Give your analysis first, then put your final score in a XML-style tag <score></score>
"""

In [5]:
from openai import OpenAI
import os

or_client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY")
)

In [6]:
import pickle
emb_path = "../datasets/human_reviews_embeddings_deepreview.pkl"
idx_path = "../datasets/human_review_score_index_deepreview.pkl"

with open(emb_path, "rb") as f:
    review_embeddings = pickle.load(f)

with open(idx_path, "rb") as f:
    review_score_index = pickle.load(f)


In [7]:
import os
from concurrent.futures import ThreadPoolExecutor
import numpy as np
bin_names = ["very_low", "low", "medium", "high", "very_high"]
thresholds = [2, 4, 6, 8]

bins = {name: [] for name in bin_names}
bins_embeddings = {name: [] for name in bin_names}

for i in review_score_index:
    score = review_score_index[i]
    idx = next((j for j, t in enumerate(thresholds) if score <= t), len(thresholds))
    name = bin_names[idx]
    bins[name].append(i)
    bins_embeddings[name].append(review_embeddings[i])

for name in bin_names:
    bins_embeddings[name] = np.array(bins_embeddings[name])

def build_sample(review):
    with open(os.path.join(reviews, review), 'r') as f:
        review_content = f.read().split("Score and Decision")[0]

    query_embedding = or_client.embeddings.create(
        model="google/gemini-embedding-001",
        input=review_content,
        encoding_format="float",
    )
    query_vector = np.array(query_embedding.data[0].embedding)

    anchor_samples = []
    for name in bin_names:
        if len(bins_embeddings[name]) == 0:
            continue
        similarities = bins_embeddings[name] @ query_vector.T
        top_indices = np.argsort(similarities)[-2:]
        selected = [bins[name][idx] for idx in top_indices]

        for filename in selected:
            if filename == review:
                continue # skip itself
            with open(os.path.join("../datasets/deepreview_13k_train/human_reviews", filename), 'r') as f:
                anchor_samples.append(f.read().split("Score and Decision")[0])

    gt = get_gt(review)
    return {
        "prompt": [
                    {
                        "content": rl_prompt,
                        "role": "system"
                    },
                    {
                        "content": f"Paper review:\n{review_content}\n\nAnchor reviews:\n" + "\n\n".join(anchor_samples),
                        "role": "user"
                    }
                ], 
        "solution": float(gt.values[0]),
        "paper_id": review.split(".")[0]
    }
import tqdm
with ThreadPoolExecutor(max_workers=50) as executor:
    ds = list(tqdm.tqdm(executor.map(build_sample, os.listdir(reviews)), total=len(os.listdir(reviews))))

100%|██████████| 978/978 [00:32<00:00, 29.79it/s]


In [8]:
from datasets import Dataset
ds_hf = Dataset.from_list(ds)
ds_hf.push_to_hub("weathon/grpo_dataset")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/weathon/grpo_dataset/commit/2d3341d3b1ad0cb094ebf8bea1d45f27eeea127c', commit_message='Upload dataset', commit_description='', oid='2d3341d3b1ad0cb094ebf8bea1d45f27eeea127c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/weathon/grpo_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='weathon/grpo_dataset'), pr_revision=None, pr_num=None)

In [9]:
import json
with open("grpo_dataset.json", "w") as f:
    json.dump(ds, f, indent=4)

In [10]:
def rollout(messages):
    _response = or_client.chat.completions.create(
        model="deepseek/deepseek-v4-flash",
        messages=messages,
        extra_body={"reasoning": {"enabled": False, "effort": "low"}, "provider": {"only": ["deepseek"]}}
    )
    response = _response.choices[0].message
    return response

In [11]:
idx = 765 
messages = ds[idx]["prompt"]

messages[0]["content"] = messages[0]["content"]#.replace("Give your analysis first, then put your final score in a XML-style tag <score></score>", "Put your final score in a XML-style tag <score></score>")

with ThreadPoolExecutor(max_workers=50) as executor:
    rollouts = list(executor.map(rollout, [messages] * 5))

In [12]:
ds[idx]["solution"]

3.0

In [13]:
rollouts

[ChatCompletionMessage(content='I need to analyze the paper under review by comparing it carefully to each anchor paper.\n\n**Paper Under Review:** A deep learning framework for turn-by-turn navigation audio instructions, with real-world A/B testing on ~600 million segments showing yaw-rate improvements over an HMM baseline. Novel multi-task formulation of audio instruction generation, cloud-edge deployment architecture.\n\n**Anchor 1: MixAttention (Scores: 1,3,3,1)** - This paper combines sliding window attention and KV cache sharing. It\'s a straightforward combination of existing techniques with no real novelty, minimal insight, and poor experiments (bar charts only). The reviews are very negative, highlighting lack of innovation and calling it a "technical report" rather than research.\n\n*Comparison*: The navigation paper has significantly more novelty (multi-task formulation of a previously rule-based problem), much stronger real-world validation (600M segments vs synthetic exper

In [14]:
print(response.content) 

NameError: name 'response' is not defined